# Province Distance Validation
Extract all 83 provinces from ProvinceData.ts and validate all pairwise distances.

## Section 1: Import Required Libraries

In [ ]:
import re
import math
from itertools import combinations
import pandas as pd

print("Libraries imported successfully")

## Section 2: Load and Parse ProvinceData.ts

In [ ]:
# Read the ProvinceData.ts file
with open('src/data/ProvinceData.ts', 'r') as f:
    content = f.read()

print(f"✓ Loaded ProvinceData.ts ({len(content)} bytes)")
print(f"✓ File contains {content.count('center:')} province definitions")

## Section 3: Extract Province Coordinates

In [ ]:
# Extract all provinces with their coordinates
# Pattern: p('id', ... center: { x: XX, y: YY } ...
pattern = r"p\('([^']+)'[^}]*center:\s*\{\s*x:\s*([\d.]+)\s*,\s*y:\s*([\d.]+)\s*\}"

matches = re.findall(pattern, content)

provinces = {}
for match in matches:
    province_id = match[0]
    x = float(match[1])
    y = float(match[2])
    provinces[province_id] = (x, y)

print(f"✓ Extracted {len(provinces)} provinces")
print(f"\nFirst 10 provinces:")
for i, (prov_id, coords) in enumerate(list(provinces.items())[:10]):
    print(f"  {prov_id:<25} {coords}")

## Section 4: Calculate Pairwise Distances

Using the Euclidean distance formula: $d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}$

In [ ]:
def euclidean_distance(p1, p2):
    """Calculate Euclidean distance between two points"""
    dx = p1[0] - p2[0]
    dy = p1[1] - p2[1]
    return math.sqrt(dx * dx + dy * dy)

# Calculate all pairwise distances
distances = []
province_ids = list(provinces.keys())

print(f"Calculating distances for {len(provinces)} provinces...")
print(f"Total pairwise combinations: {len(province_ids) * (len(province_ids) - 1) // 2}")

for id1, id2 in combinations(province_ids, 2):
    dist = euclidean_distance(provinces[id1], provinces[id2])
    distances.append({
        'province_1': id1,
        'province_2': id2,
        'distance': dist
    })

print(f"✓ Calculated {len(distances)} pairwise distances")

## Section 5: Filter and Sort Violations

In [ ]:
# Filter violations (distance < 6.1)
threshold = 6.1
violations = [d for d in distances if d['distance'] < threshold]

# Sort by distance ascending
violations_sorted = sorted(violations, key=lambda x: x['distance'])

print(f"Threshold: {threshold} units")
print(f"Total violations found: {len(violations_sorted)}")
print(f"(Pairs where distance < {threshold})")

## Section 6: Generate Validation Report

In [ ]:
print("\n" + "="*60)
print("DISTANCE VALIDATION REPORT")
print("="*60 + "\n")

if len(violations_sorted) == 0:
    print("✓ VALIDATION PASSED - All provinces properly spaced")
else:
    print(f"✗ VALIDATION FAILED - Found {len(violations_sorted)} violations\n")
    
    # Create DataFrame for better display
    df_violations = pd.DataFrame(violations_sorted)
    df_violations.columns = ['Province 1', 'Province 2', 'Distance']
    
    # Round distance for display
    df_violations['Distance'] = df_violations['Distance'].round(4)
    
    print(df_violations.to_string(index=False))
    
    print("\n" + "="*60)
    print(f"ALL {len(violations_sorted)} VIOLATIONS LISTED ABOVE (sorted by distance)")
    print("="*60)

## Section 7: Detailed Violation Report with Exact Coordinates

In [ ]:
import pandas as pd

print("\n" + "="*100)
print("COMPREHENSIVE DISTANCE VALIDATION REPORT - ALL 83 PROVINCES")
print("="*100 + "\n")

print(f"Total provinces extracted: {len(provinces)}")
print(f"Total pairwise distances calculated: {len(distances):,}")
print(f"Validation threshold: {threshold} units\n")

if len(violations_sorted) == 0:
    print("✓✓✓ VALIDATION PASSED ✓✓✓\n")
    print(f"All {len(distances):,} pairwise distances are >= {threshold} units\n")
    
    # Calculate and display statistics
    all_dists = [d['distance'] for d in distances]
    all_dists_sorted = sorted(all_dists)
    
    print("Distance Statistics:")
    print(f"  Minimum distance: {min(all_dists):.4f} units")
    print(f"  Maximum distance: {max(all_dists):.4f} units")
    print(f"  Mean distance: {sum(all_dists)/len(all_dists):.4f} units")
    print(f"  Median distance: {all_dists_sorted[len(all_dists_sorted)//2]:.4f} units")
else:
    print(f"⚠ VIOLATIONS FOUND: {len(violations_sorted)} pairs with distance < {threshold} units\n")
    
    # Create detailed violation table with exact coordinates
    print(f"All {len(violations_sorted)} violations (sorted by distance, closest first):\n")
    
    violation_details = []
    for v in violations_sorted:
        id1 = v['province_1']
        id2 = v['province_2']
        x1, y1 = provinces[id1]
        x2, y2 = provinces[id2]
        dist = v['distance']
        
        violation_details.append({
            '#': len(violation_details) + 1,
            'Province 1': id1,
            'Coords (x,y)': f"({x1:.1f}, {y1:.1f})",
            'Province 2': id2,
            'Coords (x,y) ': f"({x2:.1f}, {y2:.1f})",
            'Distance': f"{dist:.4f}"
        })
    
    df_details = pd.DataFrame(violation_details)
    print(df_details.to_string(index=False))

print("\n" + "="*100)


## Section 8: All Extracted Provinces Reference

In [ ]:
print(f"\nAll {len(provinces)} extracted provinces with coordinates:\n")

provinces_list = []
for i, (prov_id, (x, y)) in enumerate(sorted(provinces.items()), 1):
    provinces_list.append({'#': i, 'Province ID': prov_id, 'X': f"{x:.1f}", 'Y': f"{y:.1f}"})

df_provinces = pd.DataFrame(provinces_list)
print(df_provinces.to_string(index=False))

print(f"\n✓ Total: {len(provinces)} provinces")
